# FastAPI Backend Specialist - QLoRA Fine-Tuning (Colab Pro)

**MiniCPM5-1B comparison run**, per `doc/kickoff-brief-minicpm-comparison.md`.
Duplicated from the working Qwen2.5-Coder-1.5B notebook - identical dataset,
identical eval suite, identical training recipe/hyperparameters. Only the
base model (`MODEL_ID`) and output path (`OUTPUT_DIR`) differ (section 2),
plus one new verification cell (section 5a) checking whether MiniCPM5-1B's
Think/No-Think behavior engages under this pipeline's plain prompt+completion
format (no chat template is used anywhere in this notebook - see section 5a
for why that matters and what to check before committing to a full run).

The original Qwen notebook (`fastapi_lora_qlora_colab.ipynb`) is untouched -
both adapters are kept for direct comparison, per the brief's explicit
instruction not to discard either.

## 1. Install dependencies

In [ ]:
# Colab already ships torch. Pin the fine-tuning stack.
!pip install -q -U "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" \
    "bitsandbytes>=0.43" "datasets>=2.20"

## 1b. Mount Google Drive (persistence)

Colab session storage is wiped whenever the runtime disconnects (idle timeout,
closing the tab, or a natural reset) — this is more frequent on the **free**
tier than Colab Pro. Mounting Drive means the dataset only needs to be
uploaded once, and the trained adapter is saved somewhere that survives a
disconnect right after training finishes (exactly what happened in an earlier
run here).

**One-time setup:** create a folder in your Drive (e.g. `MyDrive/open_web/`)
and upload `fastapi-backend-pairs.jsonl` (and later `fastapi-eval-suite.jsonl`)
there via drive.google.com, or upload directly through the file browser on the
left after mounting below.

**Note:** mounting Drive does NOT preserve in-memory Python state (the loaded
model, tokenized dataset, etc.) across a disconnect — only files. You'll still
need to re-run the model-loading/tokenize/LoRA cells after a reconnect; what
Drive fixes is not needing to re-upload the dataset, and not losing a
finished adapter if save happens to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this to match where you put/will put your project files in Drive.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/open_web"
import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print("Drive mounted. Project dir:", DRIVE_PROJECT_DIR)

## 2. Configuration

In [ ]:
import torch

# Small base model (1-3B). Swap for another candidate (e.g. a MiniCPM / Qwen
# coder variant) when comparing base models per the architecture doc.
MODEL_ID   = "openbmb/MiniCPM5-1B"   # comparison run - was Qwen/Qwen2.5-Coder-1.5B
# Confirmed standard LlamaForCausalLM architecture (per model card) - the
# existing LoRA target_modules (section 7) should apply unchanged; verify
# via model.print_trainable_parameters() after attaching (must not be 0).

# Paths now point at the mounted Drive folder (see 1b above) so the dataset
# and the trained adapter both persist across session disconnects.
DATA_PATH  = f"{DRIVE_PROJECT_DIR}/fastapi-backend-pairs-v3.jsonl"  # contract-shaped input+output, see RESHAPE-REPORT.md
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/fastapi-backend-lora-minicpm"  # distinct path - does NOT overwrite the Qwen adapter
MAX_LEN    = 1024

# 4-bit QLoRA requires a CUDA GPU. Fall back to a CPU-friendly path if none.
USE_4BIT = torch.cuda.is_available()
print("CUDA available:", torch.cuda.is_available(), "| 4-bit QLoRA:", USE_4BIT)

## 3. Load the dataset

Uses `fastapi-backend-pairs.jsonl` if present (upload it via the Files pane),
otherwise falls back to a small embedded dummy set so the notebook always runs.

In [ ]:
import json, os

DUMMY_PAIRS = [
    {"task": "Create a FastAPI app with a GET /items endpoint that returns all items.",
     "code": "from fastapi import FastAPI\napp = FastAPI()\nitems = [{'id': 1, 'name': 'apple'}]\n\n@app.get('/items')\ndef list_items():\n    return items\n"},
    {"task": "Create a FastAPI GET /items/{item_id} endpoint returning 404 if missing.",
     "code": "from fastapi import FastAPI, HTTPException\napp = FastAPI()\nitems = {1: {'id': 1, 'name': 'apple'}}\n\n@app.get('/items/{item_id}')\ndef get_item(item_id: int):\n    if item_id not in items:\n        raise HTTPException(status_code=404, detail='Item not found')\n    return items[item_id]\n"},
    {"task": "Create a FastAPI POST /items endpoint that returns 201 with an id.",
     "code": "from fastapi import FastAPI, status\nfrom pydantic import BaseModel\napp = FastAPI()\n_db = {}\n\nclass ItemIn(BaseModel):\n    name: str\n\n@app.post('/items', status_code=status.HTTP_201_CREATED)\ndef create_item(item: ItemIn):\n    new = {'id': len(_db) + 1, 'name': item.name}\n    _db[new['id']] = new\n    return new\n"},
]

if os.path.exists(DATA_PATH):
    pairs = [json.loads(l) for l in open(DATA_PATH, encoding='utf-8')]
    print(f"Loaded {len(pairs)} real pairs from {DATA_PATH}")
else:
    pairs = DUMMY_PAIRS
    print(f"No {DATA_PATH} found — using {len(pairs)} embedded DUMMY pairs")

## 4. Prompt formatting

A simple instruction template. The frozen contract from planning (the future
input to this specialist) would replace the free-text task later; the shape
stays task -> code.

**Kept as two separate fields (prompt, completion) rather than one merged
`text` field** — this is what makes prompt-masking in section 6 possible.
The previous version merged them into one `text` string before tokenizing,
which loses the boundary needed to mask the prompt out of the loss.

In [ ]:
# v3 rows carry the REAL orchestrator prompt shape (see reshape_to_orchestrator_input.py):
# system_prompt = orchestrator.BACKEND_SYSTEM (imported from orchestrator.py, not retyped)
# user_prompt   = "LockedSpec:\n<json>\n\nDesignTokens:\n<json>" - identical f-string
#                 template to orchestrator.py:211, ground truth for what the Backend stage
#                 actually receives at inference time
# output        = the delimited-text ---FILE:---/---END--- contract completion
# Falls back to the old task/code shape only for the embedded DUMMY pairs above, which
# predate the reshape and have no system_prompt/user_prompt/output fields.

def format_example(ex):
    if "system_prompt" in ex and "user_prompt" in ex:
        prompt = ex["system_prompt"] + "\n\n" + ex["user_prompt"] + "\n\n### Response:\n"
        completion = ex["output"]
    else:
        prompt = "### Task:\n" + ex["task"] + "\n\n### FastAPI code:\n"
        completion = ex["code"]
    return {"prompt": prompt, "completion": completion}

from datasets import Dataset
raw_ds = Dataset.from_list(pairs).map(format_example)
print(raw_ds[0]["prompt"][:200])
print("---")
print(raw_ds[0]["completion"][:200])

## 5. Tokenizer & 4-bit base model (QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
else:
    # CPU fallback (no bitsandbytes) so the notebook still runs anywhere.
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

model.config.use_cache = False

## 5a. Verify Think/No-Think behavior BEFORE training

**Researched (huggingface.co/openbmb/MiniCPM5-1B model card) before writing
this cell, per the brief's explicit instruction to confirm the exact
mechanism first:** `enable_thinking` is exclusively a parameter to
`tokenizer.apply_chat_template(messages, ..., enable_thinking=False)`. The
model card documents no other control (no `/no_think` suffix or similar
raw-text marker) - Think/No-Think is a chat-template-level switch only.

**This matters here because this notebook's entire pipeline (section 4's
`format_example`, section 10's sanity check, training itself) never calls
`apply_chat_template` at all** - prompt and completion are plain string
concatenation, identical to the Qwen run (this is deliberate: it keeps the
comparison apples-to-apples, per the brief's "don't over-tune differently
per model" instruction). So the documented `enable_thinking` knob has no
hook to attach to in this pipeline as written.

**What that leaves open:** whether MiniCPM5-1B's *default* completion
behavior (no chat template, no explicit instruction either way) spontaneously
emits `<think>...</think>` reasoning anyway, as a pretrained/SFT prior. This
cell checks that empirically, on the base model, BEFORE spending compute on
training - the brief asked to confirm the mechanism "before training," and
since there's no documented raw-text equivalent to flip, an empirical check
is the correct substitute, not a guess.

**If `<think>` appears below:** LoRA fine-tuning on this dataset's
non-thinking completions (3 epochs, full loss weight on the completion
tokens) should substantially override this default for the *trained
adapter's* output even if the base checkpoint reasons by default - the
adapter learns to match our exact non-thinking completion shape. Re-check
with the section 10 sanity test after training to confirm this held. If
`<think>` does NOT appear, no further action needed - proceed as planned.

In [ ]:
# Same RAW prompt+completion style as section 4/10 (no chat template) -
# checking what the UNTRAINED base model does by default under the exact
# prompting style this notebook actually uses, not the chat-template style
# the model card's own quickstart demonstrates.
_probe_prompt = raw_ds[0]["prompt"]

_probe_inputs = tokenizer(_probe_prompt, return_tensors="pt").to(model.device)
_probe_out = model.generate(
    **_probe_inputs, max_new_tokens=120, do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)
_probe_gen = _probe_out[0][_probe_inputs["input_ids"].shape[1]:]
_probe_text = tokenizer.decode(_probe_gen, skip_special_tokens=False)

print(_probe_text[:500])
print()
if "<think>" in _probe_text:
    print("*** <think> DETECTED in raw (non-chat-template) completion - see section 5a markdown for what this means and what to check after training. ***")
else:
    print("No <think> tag detected under raw prompting - No-Think behavior confirmed by default for this pipeline's prompt style.")

## 6. Tokenize — WITH LABEL MASKING (the fix)

**Problem in the earlier version:** `labels = input_ids.copy()` trained the
model to predict the *entire* sequence, including the `### Task: ...` prompt
text and the padding tokens added to reach `MAX_LEN`. That wastes training
signal on text the model doesn't need to generate, and adds noisy gradient
signal from padding on the (likely majority of) shorter examples.

**Fix:** tokenize the prompt and completion separately, concatenate them,
then set `labels` so that:
- prompt tokens -> `-100` (ignored in the loss — the model isn't trained to
  reproduce the task description)
- completion tokens -> real token ids (this is what the model should learn
  to generate)
- padding tokens -> `-100` (ignored in the loss)

`-100` is the standard PyTorch/HF convention for "ignore this position in
`CrossEntropyLoss`".

In [ ]:
IGNORE_INDEX = -100

def tokenize_with_masking(example):
    prompt_ids = tokenizer(example["prompt"], add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(
        example["completion"], add_special_tokens=False
    )["input_ids"] + [tokenizer.eos_token_id]  # eos marks end of generation

    input_ids = prompt_ids + completion_ids
    labels = ([IGNORE_INDEX] * len(prompt_ids)) + completion_ids

    # Truncate to MAX_LEN (keep from the start; drop tail if too long)
    input_ids = input_ids[:MAX_LEN]
    labels = labels[:MAX_LEN]

    attention_mask = [1] * len(input_ids)

    # Right-pad to MAX_LEN; padding contributes to attention_mask=0 and
    # labels=-100, so it's excluded from both attention and loss.
    pad_len = MAX_LEN - len(input_ids)
    if pad_len > 0:
        input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
        attention_mask = attention_mask + [0] * pad_len
        labels = labels + [IGNORE_INDEX] * pad_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

tokenized_ds = raw_ds.map(
    tokenize_with_masking,
    remove_columns=raw_ds.column_names,
)

# Sanity check: confirm prompt + padding are masked, completion isn't.
sample = tokenized_ds[0]
n_masked = sum(1 for l in sample["labels"] if l == IGNORE_INDEX)
n_real = sum(1 for l in sample["labels"] if l != IGNORE_INDEX)
print(f"Example 0 — masked (prompt+pad) tokens: {n_masked}, trainable (completion) tokens: {n_real}")

## 7. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Train

`num_train_epochs=3` confirmed as a reasonable default on the real 71-pair
dataset (produces a healthy loss curve, ~0.87 -> ~0.07-0.13 over 54 steps).
Raise further or use the full dataset as it grows; iterate on **dataset
quality** first, hyperparameters second.

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=USE_4BIT,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds,
    data_collator=default_data_collator,
)
trainer.train()

## 9. Save the adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved to:", OUTPUT_DIR)
# The LoRA adapter is only tens of MB — download it and run base+adapter locally.

## 10. Quick inference sanity check

**Fix applied:** gradient checkpointing (enabled during training by
`prepare_model_for_kbit_training` in section 7) must be explicitly disabled
before generation, along with `model.eval()`. Leaving it enabled silently
forces `use_cache=False` during `generate()`, breaking KV caching and
producing degenerate/repeating output regardless of how well-trained the
adapter actually is — this was the root cause of the earlier garbage output
(`from 200000...`), not undertraining.

In [ ]:
# Test prompt now matches the REAL v3 training format (system_prompt + user_prompt +
# "### Response:\n"), not the old "### Task:" template. Testing with a mismatched prompt
# shape would look like a model failure when it's actually just a format mismatch.
#
# This mirrors BACKEND_SYSTEM + a real LockedSpec/DesignTokens pair from orchestrator.py -
# a genuinely held-out example (different feature/spec than anything in the training set).

model.gradient_checkpointing_disable()
model.eval()
model.config.use_cache = True

import json as _json

BACKEND_SYSTEM = 'You are the Backend specialist in a fixed software-build pipeline.\nGiven a LockedSpec and DesignTokens, write the backend code for this app.\n\nRespond using this EXACT delimited plain-text format - NOT JSON, no escaping of any kind.\nWrite your code exactly as it should appear in the file, including real newlines and quotes:\n\n---FILE: <the real path you choose, e.g. app.py>---\n<the file\'s full content, written normally, no escaping>\n---END---\n\nYou may include more than one file by repeating the block with a different real path.\n\nConcrete example (illustrative only - use your OWN real filename and implementation,\ndo not copy this example\'s filename or content):\n---FILE: example_placeholder_name.py---\ndef hello():\n    print("hi")\n\nhello()\n---END---\n\nOutput ONLY these ---FILE:---/---END--- blocks (one or more). Keep the implementation\nsimple and correct - a single main file is fine for a small app. Implement the features\nlisted in the LockedSpec, not just a placeholder/stub.'

test_locked_spec = {
    "version": 1,
    "product_type": "web_app",
    "stack": {"backend": "fastapi", "frontend": "react", "database": "postgresql", "auth": "jwt", "payments": None},
    "features": ["health check endpoint returning service status"],
}
test_design_tokens = {"colors": {"primary": "#0f172a"}, "typography": {"base": "Inter, sans-serif"}}

user_prompt = f"LockedSpec:\n{_json.dumps(test_locked_spec)}\n\nDesignTokens:\n{_json.dumps(test_design_tokens)}"
prompt = BACKEND_SYSTEM + "\n\n" + user_prompt + "\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                     pad_token_id=tokenizer.pad_token_id)
# Only print the newly generated part (the prompt itself is long now, v3-style)
gen_only = out[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(gen_only, skip_special_tokens=True))

## 11. Score against the full 31-task eval suite

Per `doc/kickoff-brief-eval-scoring.md`. Uses `eval_runner_v3.py` (multi-file-
capable `check_spec` runner, reusing `eval_runner.py`'s single-file checker and
the real `orchestrator/llm_client.extract_files` parser - **not** a
reimplementation) against `fastapi-eval-suite-v3.jsonl` (31 held-out tasks,
already in the real production prompt shape).

**Before running this cell**, upload these 3 files into this Colab session's
working directory (or the mounted Drive project folder, then `sys.path.insert`
that path) - they were built/verified locally, not in Colab:
- `eval_runner.py`
- `eval_runner_v3.py`
- `orchestrator/llm_client.py` (just this one file - `extract_files` has no
  heavy deps beyond `openai`, which is already on the Colab image via pip)

and make sure `fastapi-eval-suite-v3.jsonl` is present alongside the training
data (same `DRIVE_PROJECT_DIR`, per cell 6's `DATA_PATH` pattern).

`eval_runner_v3.py`'s checker logic (the multi-file loader, cross-file import
resolution, and check_spec assertions) was already validated locally against
hand-written correct/incorrect reference solutions before this cell was added -
see `doc/eval-scoring-report.md`. This cell only needs to supply a real
`generate_fn` from the trained model - the scoring logic itself is proven.

### 11a. Start a live Postgres instance for DB-related eval tasks

Some eval tasks (`category == "database-sqlalchemy"`) generate SQLAlchemy
apps that call `create_engine(os.getenv("DATABASE_URL", ...))` - this is
the pattern the Backend specialist was trained on after the Postgres
verification pass (`doc/postgres-verification-report.md`). For those apps
to actually connect to something real when `eval_runner_v3.py` executes
them via `TestClient`, a live Postgres instance needs to exist in this
Colab session, with `DATABASE_URL` pointed at it.

Uses **`pgserver`** - the same embedded, real-binary Postgres used for the
local Postgres-verification task, no Docker/system service needed (works
the same way in Colab: pip-installable, runs against a throwaway data
directory in the session's own filesystem).

This cell sets `EVAL_PG_BASE_URL` (not `DATABASE_URL` directly) -
`eval_runner_v3.py`'s `score_task` reads that env var and, for each eval
task, creates a **fresh, isolated database** and points `DATABASE_URL` at
it just for that task's run, then restores it afterward. This matters:
running all 31 tasks against one shared database would risk table-name
collisions and leftover rows corrupting a later task's row-count
assertions (e.g. two different db tasks both creating a `tags` table) -
isolating per task removes that risk entirely. You do not need to set
`DATABASE_URL` yourself; just run this cell before the scoring cell below.

In [ ]:
!pip install -q pgserver psycopg2-binary

import tempfile
import pgserver

_pg_data_dir = tempfile.mkdtemp(prefix="eval_pg_")
_pg_server = pgserver.get_server(_pg_data_dir)

import os as _os
_os.environ["EVAL_PG_BASE_URL"] = _pg_server.get_uri()
print("Postgres running for eval scoring:", _os.environ["EVAL_PG_BASE_URL"])
print("(eval_runner_v3.score_task will create/use a fresh isolated DB per task from this base URL)")

# To shut it down later (e.g. at the end of the notebook session):
#   _pg_server.cleanup()

In [ ]:
import sys, os

# Adjust these if you uploaded the files elsewhere in this session/Drive.
EVAL_SUPPORT_DIR = DRIVE_PROJECT_DIR  # reuse the same Drive folder as the dataset
sys.path.insert(0, EVAL_SUPPORT_DIR)

EVAL_SUITE_PATH = f"{DRIVE_PROJECT_DIR}/fastapi-eval-suite-v3.jsonl"

# Reload fresh via PeftModel so this cell works standalone in a new session too
# (not just immediately after section 8's training run in the same session).
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)
eval_model = PeftModel.from_pretrained(eval_base, OUTPUT_DIR)

# The exact fix from section 10: gradient checkpointing forces use_cache=False
# during training; must be explicitly disabled + eval() before generation.
eval_model.gradient_checkpointing_disable()
eval_model.eval()
eval_model.config.use_cache = True

def generate_fn(prompt: str) -> str:
    inputs = eval_tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    out = eval_model.generate(
        **inputs, max_new_tokens=1024, do_sample=False,
        pad_token_id=eval_tokenizer.pad_token_id,
    )
    gen_only = out[0][inputs["input_ids"].shape[1]:]
    return eval_tokenizer.decode(gen_only, skip_special_tokens=True)

from eval_runner_v3 import score_all, print_report

summary = score_all(generate_fn, EVAL_SUITE_PATH)
print_report(summary)

# Save full results for offline analysis (category clustering, cross-referencing
# against the 3 flagged dataset limitations - see doc/kickoff-brief-eval-scoring.md step 5)
import json as _json
with open(f"{DRIVE_PROJECT_DIR}/eval-results-v3.json", "w", encoding="utf-8") as f:
    _json.dump(summary, f, indent=2)
print(f"Full results saved to {DRIVE_PROJECT_DIR}/eval-results-v3.json")

## 12. Cross-reference failures against the 3 flagged dataset limitations

Per `doc/kickoff-brief-eval-scoring.md` step 5 - run after section 11 populates
`summary`. Checks whether failures cluster around: (a) DB-related tasks
(SQLite/Postgres gap), (b) implicit multi-feature specs (the dataset's
single-feature-list training bias), (c) design_tokens content having any
visible effect on backend output (expected: little to none, backend code
shouldn't depend on visual tokens).

In [ ]:
db_keywords = ("db", "database", "sql", "postgres", "sqlite", "persist", "store")
db_related = [r for r in summary["results"] if any(k in r["task"].lower() for k in db_keywords)]
print(f"DB-related tasks: {sum(r['passed'] for r in db_related)}/{len(db_related)} passed")
for r in db_related:
    print(f"  [{'PASS' if r['passed'] else 'FAIL'}] {r['task'][:70]}")

print()
non_db = [r for r in summary["results"] if r not in db_related]
print(f"Non-DB tasks: {sum(r['passed'] for r in non_db)}/{len(non_db)} passed")
print("(Compare the two pass rates above - a notably lower DB pass rate would")
print(" support the SQLite/Postgres training-gap hypothesis flagged in architecture.md)")

## Notes for scaling up

- Upload the full `fastapi-backend-pairs.jsonl` (cell 3 auto-detects it).
- Compare base models (Qwen coder / MiniCPM etc.) by re-running with a different
  `MODEL_ID` and scoring each against `fastapi-eval-suite.jsonl` using
  `eval_runner.py`.
- Iterate on **dataset quality** first, hyperparameters second (per the brief).
- Keep `use_cache=False` during training, `True` for generation (handled above).
- **Label masking (section 6) must be preserved** in any future edits to the
  tokenization step — reverting to `labels = input_ids.copy()` (or reintroducing
  `DataCollatorForLanguageModeling`, which recomputes labels the same way)
  silently undoes the fix.